# HeH⁺ potential energy surface: from exact references to real quantum hardware
**Team:** *Name A* (algorithm & physics), *Name B* (hardware & noise)

**BasQ Qiskit Fall Fest 2026, Challenge 1, Beginner track**

## Abstract
*TODO (both, last 10 min): 4–5 sentences. System, what we pushed beyond the baseline, main numbers (equilibrium length, D_e, best hardware error vs exact), and the main limitation.*

## How this notebook scores against the judging criteria
| Criterion | Where we address it |
|---|---|
| **Technical complexity** | Section 1 qubit tapering 4→2 qubits; Section 4 larger basis (6-31G), ansatz and optimizer study |
| **Validation** | Section 2 independent PySCF FCI cross-check; Section 3 VQE vs exact over the full curve; Section 6 noisy simulation; Section 7 real hardware; Section 8 all levels compared |
| **Hardware awareness** | Section 5 encoding vs routing depth, noise-aware layout selection; Section 6 ZNE; Section 7 dynamical decoupling, twirling, TREX on a real device |
| **Performance and limitations** | Section 8 error budget (ansatz / shots / noise / mitigation residual) and scaling |
| **Presentation** | Reproducibility below; each section states its claim, evidence and caveat |

## Reproducibility
```bash
pip install -r requirements.txt          # pinned versions
python beginner/check_setup.py           # must print OK
```
Run top to bottom (≈ *TODO* min without hardware). Hardware results are loaded from `results/hardware_*.csv` unless `RUN_HARDWARE = True`. All energies are **total energies in Hartree**. Chemical accuracy = 1.6 mHa.

## Team plan (delete this cell before submitting)

| Time | A: algorithm & physics | B: hardware & noise |
|---|---|---|
| 0:00–0:10 | Read the template together, run Section 0, agree on the distance grid | same |
| 0:10–0:50 | Sections 1, 2, 3 (noiseless PES, equilibrium, dissociation) | Section 5 transpile study, then **submit the hardware job by ~0:45** (Section 7) because of queue time |
| 0:50–1:25 | Section 4 beyond the baseline (6-31G, ansatz/optimizer) | Section 6 noisy PES and ZNE; collect hardware results when they're ready |
| 1:25–1:45 | Section 8 error budget and combined figure (A writes it, B supplies numbers) | Section 7 hardware analysis, Section 8 noise rows |
| 1:45–2:00 | Abstract and conclusions, **Restart & Run All**, export HTML | Proofread, check figures and job IDs, final commit |

**Git rules for the next two hours:** only edit cells in your own sections. Experiment in `scratch_<name>.ipynb`. Every section saves `results/*.csv` and `figures/*.png`. Pull before you start, and commit and push every ~15 min.

# 0. Setup

In [ ]:
import time, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from pes_tools import (CHEMICAL_ACCURACY, HARTREE_TO_KCAL_MOL, build_problem,
                       hf_energy, exact_energy, vqe_energy, uccsd_ansatz)
from hardware_tools import (make_mapper, prepare_point, transpile_report, best_transpile,
                            noisy_estimator, estimate, fold_two_qubit_gates, zne_extrapolate,
                            runtime_estimator)

warnings.filterwarnings("ignore", module="scipy")
RESULTS, FIGURES = Path("results"), Path("figures")
RUN_HARDWARE = False  # flip to True only to (re)submit real-device jobs

# Shared distance grid (Angstrom). Agree on this in the first 10 minutes.
DISTANCES = np.round(np.concatenate([np.linspace(0.5, 1.2, 8), np.linspace(1.4, 3.5, 6)]), 3)  # TODO
HW_DISTANCES = [0.6, 0.774, 1.0, 1.5, 2.5]  # TODO: subset for noisy sim / hardware

# 1. System and qubit encoding
- **Owner:** A
- **Scores on:** Technical complexity, Presentation
- **Timebox:** 0:10–0:20

HeH⁺: 2 electrons (He 2 + H 1 − 1 for the charge), closed-shell singlet. In STO-3G that is 2 spatial orbitals, i.e. 4 spin-orbitals.

**Claim:** exploiting the electron-number symmetries shrinks the problem from 4 qubits (Jordan-Wigner) to 2 (parity mapping + two-qubit reduction) with **no loss of accuracy**, and that is what makes the hardware runs in Sections 5–7 feasible.

**Why the reduction is exact.** The molecular Hamiltonian conserves the number of spin-up electrons N↑ and spin-down electrons N↓. In the parity encoding, qubit *k* stores the parity of the occupations of spin-orbitals 0…*k*, so the qubit at the end of the spin-up block stores (−1)^N↑ and the last qubit stores (−1)^(N↑+N↓). Both are fixed numbers for our molecule (N↑ = N↓ = 1), so those two qubits are never flipped by the Hamiltonian: every Pauli term acts on them with only I or Z. We replace those Z's by their known eigenvalues (±1) and delete the two qubits. The 2-qubit Hamiltonian has exactly the same spectrum as the 4-qubit one *inside* the correct symmetry sector, which is the sector we want. This is the ℤ₂ "qubit tapering" of Bravyi, Gambetta, Mezzacapo & Temme, *arXiv:1701.08213* (2017).

**Why HeH⁺ stops at 2 qubits while H₂ goes down to 1.** Qiskit's tapering also searches for any *extra* ℤ₂ symmetries of the qubit Hamiltonian. H₂ has one, inversion symmetry (the two atoms are identical), so it tapers to a single qubit. HeH⁺ is heteronuclear, so there is no inversion centre and no extra symmetry: `tapered` gives the same 2 qubits as `parity` (checked in the cell below).

In [ ]:
problem = build_problem(0.774)
rows = []
for kind in ["jw", "parity", "tapered"]:
    m = make_mapper(problem, kind)
    op = m.map(problem.hamiltonian.second_q_op())
    a = uccsd_ansatz(problem, m)
    rows.append({"encoding": kind, "qubits": op.num_qubits, "pauli_terms": len(op),
                 "uccsd_params": a.num_parameters, "exact_Ha": exact_energy(problem, m)})
encodings = pd.DataFrame(rows); encodings

In [ ]:
# Same pipeline on H2 (identical atoms -> extra inversion symmetry) for contrast
from qiskit_nature.second_q.drivers import PySCFDriver
h2 = PySCFDriver(atom="H 0 0 0; H 0 0 0.735", basis="sto3g").run()
print("H2   qubits:", {k: make_mapper(h2, k).map(h2.hamiltonian.second_q_op()).num_qubits for k in ["jw", "parity", "tapered"]})
print("HeH+ qubits:", dict(zip(encodings.encoding, encodings.qubits)))
print("Max spread of exact energies across encodings:", f"{np.ptp(encodings.exact_Ha):.1e} Ha")
print("\nThe whole 2-qubit HeH+ Hamiltonian at 0.774 Å (electronic part, Ha):")
print(make_mapper(problem, "tapered").map(problem.hamiltonian.second_q_op()))

**Reading the table.** Jordan-Wigner needs 4 qubits and 27 Pauli terms; the 2-qubit encoding needs only 9 terms, and the exact energy is the same in all three encodings (identical to machine precision). That confirms the reduction is lossless. Fewer qubits mean shallower routed circuits (Section 5), and fewer Pauli terms mean fewer measurement settings on hardware. UCCSD always has 3 parameters here: two single excitations (one per spin) and one double excitation that moves both electrons from the bonding to the antibonding orbital. `parity` and `tapered` coincide because HeH⁺ has no spatial symmetry left to remove, unlike H₂ (1 qubit).

# 2. Classical references and an independent check
- **Owner:** A
- **Scores on:** Validation
- **Timebox:** 0:20–0:25

Before trusting any quantum number, we check our "exact" reference two independent ways: Qiskit Nature's diagonalisation and PySCF's own FCI solver.

**Pitfall we found:** the tutorial's unfiltered `NumPyMinimumEigensolver` searches every electron count. For HeH⁺ it returns a state with extra electrons, ≈0.16 Ha too low. We filter on particle number and spin (`exact_energy` in `pes_tools.py`).

In [ ]:
from pyscf import gto, scf, fci
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

def pyscf_fci(d, basis="sto3g"):
    mol = gto.M(atom=f"He 0 0 0; H 0 0 {d}", basis=basis, charge=1, spin=0, verbose=0)
    mf = scf.RHF(mol).run()
    return mf.e_tot, fci.FCI(mf).kernel()[0]

e_rhf, e_fci = pyscf_fci(0.774)
p = build_problem(0.774)
unf = GroundStateEigensolver(make_mapper(p, "jw"), NumPyMinimumEigensolver()).solve(p)
unfiltered = unf.total_energies[0]
print(f"PySCF RHF          {e_rhf: .8f}   Qiskit HF circuit {hf_energy(p): .8f}")
print(f"PySCF FCI          {e_fci: .8f}   Qiskit exact      {exact_energy(p): .8f}")
print(f"Unfiltered (wrong) {unfiltered: .8f}   off by {unfiltered - e_fci:+.3f} Ha, "
      f"electrons in that state = {unf.num_particles[0]:.0f} (should be 2)")

# Same cross-check over the whole distance grid, not just one point
check = pd.DataFrame([{"distance_A": d, "pyscf_fci": pyscf_fci(d)[1],
                       "qiskit_exact": exact_energy(build_problem(float(d)), make_mapper(build_problem(float(d)), "tapered"))}
                      for d in DISTANCES])
check["diff"] = (check.pyscf_fci - check.qiskit_exact).abs()
check.to_csv(RESULTS / "fci_crosscheck.csv", index=False)
print(f"\nMax |PySCF FCI - Qiskit tapered exact| over {len(check)} distances: {check['diff'].max():.1e} Ha")
assert check["diff"].max() < 1e-8

**What this shows.** Two independent codes (PySCF's FCI solver and Qiskit's diagonalisation of the *tapered 2-qubit* Hamiltonian) agree to ~10⁻¹⁴ Ha at every distance. That validates the reference, the integrals, the fermion-to-qubit mapping and the tapering in one go. Qiskit's HF circuit also reproduces PySCF's RHF energy exactly, so the starting point of VQE is right.

The unfiltered solver returns a state with **3 electrons** (neutral HeH, about 0.16 Ha too low). The qubit Hamiltonian lives in Fock space, where all electron numbers coexist, and more electrons can mean lower energy. It went unnoticed in the H₂ tutorial only because there the 2-electron state happened to be the lowest. Every "exact" number in this notebook is filtered (or tapered, which fixes the electron number by construction).

# 3. Noiseless potential energy surface
- **Owner:** A
- **Scores on:** Validation, Performance
- **Timebox:** 0:25–0:50

Noiseless VQE (UCCSD, tapered 2 qubits, SLSQP) against HF and exact over the whole curve. Each point is warm-started from the previous one's parameters, and Section 4c compares that with cold starts.

In [ ]:
rows, prev = [], None
for d in DISTANCES:
    prob = build_problem(float(d)); m = make_mapper(prob, "tapered")
    v = vqe_energy(prob, mapper=m, initial_point=prev)
    prev = v.optimal_point
    rows.append({"distance_A": d, "hf": hf_energy(prob, m), "vqe": v.energy,
                 "exact": exact_energy(prob, m), "evals": v.cost_function_evals})
pes = pd.DataFrame(rows)
pes["vqe_err"] = (pes.vqe - pes.exact).abs()
pes["correlation"] = pes.exact - pes.hf
pes.to_csv(RESULTS / "pes_noiseless_sto3g.csv", index=False)
pes

In [ ]:
fig, (ax, ax2, ax3) = plt.subplots(3, 1, figsize=(8, 9), sharex=True, height_ratios=[3, 1.2, 1.2])
ax.plot(pes.distance_A, pes.hf, "o--", c="#9e9e9e", label="Hartree-Fock")
ax.plot(pes.distance_A, pes.exact, "-", c="#2ca02c", label="Exact (FCI)")
ax.plot(pes.distance_A, pes.vqe, "x", c="#1f77b4", ms=9, label="VQE (UCCSD, 2 qubits)")
ax.set_ylabel("Total energy (Ha)"); ax.legend(); ax.set_title("HeH$^+$ PES, STO-3G, noiseless")
ax2.semilogy(pes.distance_A, pes.vqe_err, "o-", c="#1f77b4", label="|VQE − exact|")
ax2.axhline(CHEMICAL_ACCURACY, c="red", ls=":", label="chemical accuracy")
ax2.set_ylim(1e-12, 1e-2); ax2.set_ylabel("Error (Ha)"); ax2.legend(loc="lower right", fontsize=8)
ax3.plot(pes.distance_A, 1e3 * pes.correlation, "s-", c="#d62728")
ax3.axhline(-1e3 * CHEMICAL_ACCURACY, c="red", ls=":")
ax3.set_ylabel("E$_{corr}$ (mHa)"); ax3.set_xlabel("He–H distance (Å)")
fig.tight_layout(); fig.savefig(FIGURES / "pes_noiseless_sto3g.png", dpi=150)

### 3a. Equilibrium bond length

In [ ]:
from scipy.interpolate import CubicSpline
from scipy.optimize import minimize_scalar
spl = CubicSpline(pes.distance_A, pes.vqe)
opt = minimize_scalar(spl, bounds=(pes.distance_A.min(), pes.distance_A.max()), method="bounded")
r_spline = opt.x
print(f"r_eq (VQE, spline over grid) = {r_spline:.4f} Å")

# Refine: minimise the VQE energy itself (one VQE run per trial distance), warm-started
p0 = build_problem(r_spline)
warm = vqe_energy(p0, mapper=make_mapper(p0, "tapered")).optimal_point   # warm start for the refinement
def vqe_at(d):
    prob = build_problem(float(d))
    return vqe_energy(prob, mapper=make_mapper(prob, "tapered"), initial_point=warm).energy
ref = minimize_scalar(vqe_at, bounds=(r_spline - 0.05, r_spline + 0.05), method="bounded", options={"xatol": 1e-4})
r_eq, e_eq = ref.x, float(ref.fun)
p_eq = build_problem(r_eq)
e_eq_exact = exact_energy(p_eq, make_mapper(p_eq, "tapered"))
R_EQ_LIT, DE_LIT_EV = 0.7743, 2.040   # Kolos & Peek, Chem. Phys. 12, 381 (1976): R_e = 1.4632 bohr, D_e = 2.040 eV
print(f"r_eq (VQE, direct)           = {r_eq:.4f} Å,  E_VQE = {e_eq:.6f} Ha,  |E_VQE - E_exact| = {abs(e_eq - e_eq_exact):.1e} Ha")
print(f"Literature (near-exact)      = {R_EQ_LIT:.4f} Å   ->  STO-3G error = {r_eq - R_EQ_LIT:+.3f} Å ({100 * (r_eq / R_EQ_LIT - 1):+.0f} %)")

### 3b. Dissociation

In [ ]:
def atom_fci(symbol, charge, spin, basis="sto3g"):
    nelec = {"He": 2, "H": 1}[symbol] - charge
    if nelec == 0:
        return 0.0  # a bare proton has no electrons
    mol = gto.M(atom=f"{symbol} 0 0 0", basis=basis, charge=charge, spin=spin, verbose=0)
    mf = (scf.RHF if spin == 0 else scf.ROHF)(mol).run()
    return fci.FCI(mf).kernel()[0]

channels = {"He + H+": atom_fci("He", 0, 0) + atom_fci("H", 1, 0),
            "He+ + H": atom_fci("He", 1, 1) + atom_fci("H", 0, 1)}
for k, v in channels.items(): print(f"{k:8s} {v: .6f} Ha")
print(f"Gap between channels: {channels['He+ + H'] - channels['He + H+']:.3f} Ha "
      f"(real gap = IE(He) - IE(H) = 24.59 - 13.60 eV = {(24.587 - 13.598) / 27.2114:.3f} Ha)")
e_inf = min(channels.values())
print(f"Long-distance end of our curve: {pes.exact.iloc[-1]:.6f} Ha  "
      f"(only {1e6 * (e_inf - pes.exact.iloc[-1]):.1f} µHa below He + H+, so converged at {pes.distance_A.iloc[-1]} Å)")
D_e = e_inf - e_eq
print(f"D_e (STO-3G, VQE) = {D_e:.4f} Ha = {D_e * HARTREE_TO_KCAL_MOL:.1f} kcal/mol = {D_e * 27.2114:.2f} eV   "
      f"vs literature {DE_LIT_EV} eV")

**Interpretation of Section 3.**
- **VQE is essentially exact on the noiseless simulator.** The worst |VQE − exact| over the 14 distances is ~3×10⁻⁷ Ha, about 5000× below chemical accuracy. The tapered space has only 4 states and UCCSD's 3 rotations can reach the exact ground state in it, so the only error left is the optimiser's stopping tolerance. Warm-starting keeps every point to ≤ 18 energy evaluations.
- **Correlation energy is small and lives at short distance.** E_exact − E_HF peaks at about −10 mHa around 0.7 Å, which is six times chemical accuracy, so Hartree-Fock alone would *not* be good enough near equilibrium. It falls below 1.6 mHa beyond ~1.5 Å and vanishes at dissociation.
- **Why it vanishes (and why this differs from H₂).** HeH⁺ dissociates into He + H⁺: a closed-shell atom plus a bare proton. In STO-3G He has a single 1s orbital, so its two electrons have nowhere else to go and HF is exact there. For H₂, restricted HF fails badly at dissociation because it forces ionic H⁻ + H⁺ character. HeH⁺ is a "well-behaved" case for HF, and correlation matters mostly in the bonding region.
- **Dissociation channel.** The electrons stay on helium: He + H⁺ is lower than He⁺ + H because helium holds its electrons much more tightly (ionisation energy 24.6 eV vs 13.6 eV). STO-3G gets the order right. The curve is flat to a few µHa by 3.5 Å, so our grid reaches the asymptote *in this basis*. In reality, H⁺ polarises He and gives an attractive −α/(2R⁴) tail (≈ −0.4 mHa at 3.5 Å), which a basis with no p-functions cannot describe.
- **The real limitation is the basis, not the quantum algorithm.** STO-3G puts the minimum at ≈ 0.91 Å (+18 % vs the true 0.774 Å) and underestimates D_e (≈ 1.5 eV vs 2.04 eV). Section 4 shows these errors are ~10⁵× larger than the VQE error and that a bigger basis removes them.

# 4. Beyond the baseline
- **Owner:** A
- **Scores on:** Technical complexity, Performance
- **Timebox:** 0:50–1:25

Three studies, each answering one question:
- **4a. Basis set.** How much of our error comes from STO-3G itself? We run VQE in 6-31G (4 spatial orbitals → 6 tapered qubits, 15 UCCSD parameters) over a coarse grid, then compare with a purely classical FCI "basis-set ladder" up to cc-pVQZ to reach the near-exact answer.
- **4b. Ansatz.** UCCSD (chemistry-inspired) vs hardware-efficient circuits (`real_amplitudes`, `efficient_su2`): parameters, CNOT count, and robustness to random starting points.
- **4c. Optimiser.** SLSQP vs COBYLA vs SPSA, noiseless and with simulated shot noise, plus warm vs cold starts along the curve.

In [ ]:
rows = []
for basis in ["sto3g", "6-31g"]:
    for kind in ["jw", "tapered"]:
        t0 = time.perf_counter()
        prob = build_problem(0.774, basis=basis); m = make_mapper(prob, kind)
        v = vqe_energy(prob, mapper=m)
        rows.append({"basis": basis, "encoding": kind, "qubits": v.num_qubits, "params": v.num_parameters,
                     "vqe": v.energy, "exact": exact_energy(prob, m), "seconds": time.perf_counter() - t0})
scaling = pd.DataFrame(rows); scaling["err"] = (scaling.vqe - scaling.exact).abs()
scaling.to_csv(RESULTS / "basis_encoding_scaling.csv", index=False); scaling

In [ ]:
# 4a. 6-31G potential energy surface with VQE (6 tapered qubits, 15 parameters)
GRID_631G = [0.6, 0.7, 0.75, 0.8, 0.9, 1.1, 1.5, 2.5, 3.5]
rows, prev = [], None
for d in GRID_631G:
    t0 = time.perf_counter()
    prob = build_problem(d, basis="6-31g"); m = make_mapper(prob, "tapered")
    v = vqe_energy(prob, mapper=m, initial_point=prev); prev = v.optimal_point
    rows.append({"distance_A": d, "hf": hf_energy(prob, m), "vqe": v.energy, "exact": exact_energy(prob, m),
                 "evals": v.cost_function_evals, "seconds": time.perf_counter() - t0})
pes631 = pd.DataFrame(rows)
pes631["vqe_err"] = (pes631.vqe - pes631.exact).abs(); pes631["correlation"] = pes631.exact - pes631.hf
pes631.to_csv(RESULTS / "pes_noiseless_631g.csv", index=False)
r631 = minimize_scalar(CubicSpline(pes631.distance_A, pes631.vqe), bounds=(0.6, 1.1), method="bounded")
De631 = atom_fci("He", 0, 0, basis="6-31g") - r631.fun
print(f"6-31G VQE: r_eq = {r631.x:.4f} Å, D_e = {De631 * 27.2114:.2f} eV, max |VQE-exact| = {pes631.vqe_err.max():.1e} Ha, "
      f"~{pes631.seconds.mean():.1f} s/point")
pes631

In [ ]:
# 4a (cont.). Classical FCI basis-set ladder: where does the true answer lie?
def fci_total(d, basis):
    mol = gto.M(atom=f"He 0 0 0; H 0 0 {d}", basis=basis, charge=1, spin=0, verbose=0)
    return fci.FCI(scf.RHF(mol).run()).kernel()[0], mol.nao

rows = []
for basis in ["sto3g", "6-31g", "6-31g**", "cc-pvdz", "cc-pvtz", "cc-pvqz"]:
    o = minimize_scalar(lambda d: fci_total(d, basis)[0], bounds=(0.6, 1.1), method="bounded", options={"xatol": 1e-4})
    nao = fci_total(0.8, basis)[1]
    De = atom_fci("He", 0, 0, basis=basis) - o.fun
    rows.append({"basis": basis, "spatial_orbitals": nao, "qubits_JW": 2 * nao, "qubits_tapered": 2 * nao - 2,
                 "r_eq_A": o.x, "E_eq_Ha": o.fun, "D_e_eV": De * 27.2114})
ladder = pd.DataFrame(rows)
ladder["r_eq_err_A"] = ladder.r_eq_A - R_EQ_LIT; ladder["D_e_err_eV"] = ladder.D_e_eV - DE_LIT_EV
ladder.to_csv(RESULTS / "basis_ladder_fci.csv", index=False)
print(f"Literature: r_eq = {R_EQ_LIT} Å, D_e = {DE_LIT_EV} eV (Kolos & Peek 1976)")
ladder.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for df, lab, c in [(pes, "STO-3G (2 qubits)", "#1f77b4"), (pes631, "6-31G (6 qubits)", "#ff7f0e")]:
    ax.plot(df.distance_A, df.exact, "-", c=c, alpha=0.6)
    ax.plot(df.distance_A, df.vqe, "o", c=c, label=f"VQE {lab}")
    ax.plot(df.distance_A, df.hf, "--", c=c, alpha=0.4)
qz = ladder.set_index("basis").loc["cc-pvqz"]
ax.plot(qz.r_eq_A, qz.E_eq_Ha, "k*", ms=14, label="FCI cc-pVQZ minimum (classical)")
ax.axvline(R_EQ_LIT, c="k", ls=":", lw=1, label="literature r$_{eq}$")
ax.set_xlabel("He–H distance (Å)"); ax.set_ylabel("Total energy (Ha)")
ax.set_title("Basis set dominates the error: VQE (dots) sits on exact (lines), HF dashed"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGURES / "pes_basis_comparison.png", dpi=150)

### 4b. Ansatz comparison (STO-3G, tapered, at our r_eq)

In [ ]:
from qiskit import transpile
from qiskit.circuit.library import efficient_su2, real_amplitudes
from qiskit_nature.second_q.circuit.library import HartreeFock

prob = build_problem(r_eq); m = make_mapper(prob, "tapered"); e_ref = exact_energy(prob, m)
hf_circ = HartreeFock(prob.num_spatial_orbitals, prob.num_particles, m)
ansatze = {"UCCSD": uccsd_ansatz(prob, m),
           "real_amplitudes reps=1": hf_circ.compose(real_amplitudes(2, reps=1)),
           "efficient_su2 reps=1": hf_circ.compose(efficient_su2(2, reps=1)),
           "efficient_su2 reps=2": hf_circ.compose(efficient_su2(2, reps=2))}
rng = np.random.default_rng(7)
N_RANDOM = 4
rows = []
for name, a in ansatze.items():
    flat = transpile(a.decompose(reps=3), basis_gates=["cx", "rz", "sx", "x"], optimization_level=1)
    errs, evals = [], []
    for trial in range(1 + N_RANDOM):   # trial 0 starts at zero parameters, the rest at random points
        x0 = np.zeros(a.num_parameters) if trial == 0 else rng.uniform(-np.pi, np.pi, a.num_parameters)
        v = vqe_energy(prob, mapper=m, ansatz=a, initial_point=x0)
        errs.append(abs(v.energy - e_ref)); evals.append(v.cost_function_evals)
    rows.append({"ansatz": name, "params": a.num_parameters, "cnots": flat.count_ops().get("cx", 0), "depth": flat.depth(),
                 "err_zero_start": errs[0], "worst_err_random_start": max(errs[1:]), "mean_evals": np.mean(evals)})
ansatz_cmp = pd.DataFrame(rows); ansatz_cmp.to_csv(RESULTS / "ansatz_comparison.csv", index=False); ansatz_cmp

### 4c. Optimiser comparison under shot noise, and warm vs cold starts

On hardware every energy is an *estimate* from finite shots. We mimic that with `StatevectorEstimator(default_precision=σ)`, which adds Gaussian noise of standard deviation σ (Ha) to every energy evaluation. σ = 0.001 Ha needs roughly 10⁵–10⁶ shots per evaluation for this Hamiltonian. We score each optimiser by the **true** (noiseless) energy at the parameters it returns, not by the noisy value it reports.

*Implementation detail that bit us:* passing an integer `seed` makes the estimator restart its random generator on every call, so the "noise" becomes the same constant offset each time and cannot disturb the optimiser. We pass a persistent `np.random.Generator` instead.

In [ ]:
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.optimizers import SLSQP, COBYLA, SPSA
from qiskit_algorithms.utils import algorithm_globals

a = uccsd_ansatz(prob, m); op = m.map(prob.hamiltonian.second_q_op())
true_energy = lambda x: float(StatevectorEstimator().run([(a, op, x)]).result()[0].data.evs) + prob.nuclear_repulsion_energy
optimizers = {"SLSQP": lambda: SLSQP(), "COBYLA": lambda: COBYLA(maxiter=200), "SPSA": lambda: SPSA(maxiter=300)}
rows = []
for sigma in [0.0, 0.001, 0.005]:
    for name, make in optimizers.items():
        for seed in range(3 if sigma else 1):
            algorithm_globals.random_seed = seed          # SPSA's own random perturbations
            est = StatevectorEstimator(default_precision=sigma, seed=np.random.default_rng(seed))
            v = vqe_energy(prob, mapper=m, ansatz=a, optimizer=make(), estimator=est)
            rows.append({"sigma_Ha": sigma, "optimizer": name, "seed": seed, "evals": v.cost_function_evals,
                         "true_err": abs(true_energy(v.optimal_point) - e_ref), "reported_minus_exact": v.energy - e_ref})
opt_cmp = pd.DataFrame(rows); opt_cmp.to_csv(RESULTS / "optimizer_comparison.csv", index=False)
summary = opt_cmp.groupby(["sigma_Ha", "optimizer"]).agg(evals=("evals", "mean"), median_true_err=("true_err", "median"),
                                                         worst_true_err=("true_err", "max"),
                                                         mean_reported_minus_exact=("reported_minus_exact", "mean"))
summary["within_chem_acc"] = summary.worst_true_err < CHEMICAL_ACCURACY
summary

In [ ]:
# Warm vs cold starts along the whole STO-3G curve (Section 3 used warm starts)
cold = []
for d in DISTANCES:
    prob_d = build_problem(float(d)); v = vqe_energy(prob_d, mapper=make_mapper(prob_d, "tapered"))
    cold.append({"distance_A": d, "cold_evals": v.cost_function_evals, "cold_vqe": v.energy})
warm_cold = pes[["distance_A", "evals", "vqe"]].rename(columns={"evals": "warm_evals", "vqe": "warm_vqe"}).merge(pd.DataFrame(cold))
warm_cold.to_csv(RESULTS / "warm_vs_cold.csv", index=False)
print(f"Total evaluations over {len(DISTANCES)} points: warm {warm_cold.warm_evals.sum()}, cold {warm_cold.cold_evals.sum()} "
      f"({100 * (1 - warm_cold.warm_evals.sum() / warm_cold.cold_evals.sum()):.0f} % saved); "
      f"max energy difference {np.abs(warm_cold.warm_vqe - warm_cold.cold_vqe).max():.1e} Ha")

**Interpretation of Section 4.**
- **Basis-set error dwarfs everything else.** At equilibrium, going from STO-3G to 6-31G lowers the energy by ~80 mHa and moves r_eq from ≈ 0.91 Å to ≈ 0.77 Å, while VQE stays within ~10⁻⁷ Ha of exact in *both* bases. So for a noiseless simulation the quantum algorithm is not the bottleneck; the basis is. The classical ladder converges to r_eq ≈ 0.775 Å and D_e ≈ 2.04 eV in cc-pVQZ, matching the literature (0.774 Å, 2.04 eV) and independently validating those reference values.
- **The STO-3G picture of correlation was partly an artefact.** In 6-31G the correlation energy is about −22 mHa near equilibrium (twice STO-3G) and does *not* vanish at dissociation (−15 mHa at 3.5 Å): helium now has a second s orbital, so its own two electrons can correlate. In a minimal basis an isolated He atom has zero correlation energy; the real value is about −42 mHa.
- **Polarisation functions matter for binding.** 6-31G already fixes r_eq but still underestimates D_e (≈ 1.7 eV). Adding p-functions (6-31G**) brings D_e to ≈ 2.0 eV: the bond in HeH⁺ is largely the proton polarising helium's electron cloud, which s-only bases cannot describe.
- **The price is qubits and run time.** 6-31G needs 6 tapered qubits (8 in JW) and 15 UCCSD parameters; each VQE point costs ~70–120 evaluations and 5–9 s instead of ~15 evaluations and 0.2 s. Tapering also pays off classically: the 8-qubit JW run takes ~30 s vs ~9 s tapered. cc-pVQZ would need 118 tapered qubits, far beyond what VQE on today's hardware can handle, while classical FCI does it in seconds for 2 electrons. *HeH⁺ is a validation problem, not one where a quantum computer has an advantage.*
- **Ansatz:** all four ansätze reach the exact energy from a zero start. The hardware-efficient `real_amplitudes` circuit needs only **1 CNOT** (UCCSD: 4 after compilation), because any real 2-qubit state can be made with one CNOT, and after tapering every 2-qubit state already has the right electron number, so a hardware-efficient ansatz cannot "leak" into unphysical states here. The cost is optimisation effort: `efficient_su2` has 8–12 parameters, needs 5–7× more evaluations than UCCSD, and from random starts the reps=1 version stopped ~10⁻⁴ Ha short of the minimum (still within chemical accuracy here). That is the classic trainability trade-off of hardware-efficient circuits, and it worsens quickly with size. (Hint for B: the 1-CNOT circuit is the lowest-noise choice for hardware.)
- **Optimiser under shot noise:** noiselessly, SLSQP is the fastest (~14 evaluations). With shot noise it **fails badly** (errors of ~1 Ha): its finite-difference gradients use tiny steps, so they are pure noise. Gradient-free COBYLA uses few evaluations but stops once noise hides further progress, ending a few mHa off (median ≈ 2 mHa at σ = 1 mHa, worst ≈ 9 mHa at σ = 5 mHa), so it misses chemical accuracy. SPSA, designed for noisy objectives, was the only optimiser within chemical accuracy in every noisy run (worst ≈ 0.9 mHa), at the price of 600 evaluations.
- **Winner's curse:** under noise, the *reported* VQE energy is often **below** the exact ground state. That seems to violate the variational principle, but it is a selection effect: the optimiser keeps whichever noisy estimate happened to be lowest. We saw it for COBYLA at σ = 5 mHa (reported ≈ 6 mHa *below* exact on average). On hardware, always re-measure the final parameters with fresh shots.
- **Warm starts barely help here:** 169 vs 176 evaluations over the curve (4 %), with identical energies. The UCCSD amplitudes are tiny (|θ| < 0.07 rad), so the all-zero Hartree-Fock start is already close to the optimum. Warm starts should matter more for larger bases and on hardware, where every evaluation costs shots.

# 5. Hardware-aware compilation
- **Owner:** B
- **Scores on:** Hardware awareness
- **Timebox:** 0:10–0:35

**Claim:** the choice of encoding sets the routed circuit depth on a heavy-hex device, and the depth sets the noise. We compare encodings and transpiler optimisation levels, then pick the physical layout with the lowest total reported error (gate + readout errors from the backend calibration), scanning transpiler seeds.

Backend: `FakeFez` (a snapshot of IBM Heron r2 calibration). *TODO (B): switch to the same device family as the real run in Section 7.*

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeFez
backend = FakeFez()

rows = []
for kind in ["jw", "tapered"]:
    pp = prepare_point(0.774, mapping=kind)
    for lvl in [0, 1, 2, 3]:
        r = transpile_report(pp.circuit, backend, optimization_level=lvl, seed=0)
        rows.append({"encoding": kind, "opt_level": lvl, "depth": r["depth"],
                     "two_qubit_gates": r["two_qubit_gates"], "error_score": r["error_score"]})
compile_study = pd.DataFrame(rows)
compile_study.to_csv(RESULTS / "compile_study.csv", index=False); compile_study

In [ ]:
pp_eq = prepare_point(0.774, mapping="tapered")
best = best_transpile(pp_eq.circuit, backend, n_seeds=20)
print("chosen physical qubits:", best["physical_qubits"], "| depth", best["depth"],
      "| 2q gates", best["two_qubit_gates"], "| error score", round(best["error_score"], 4))
best["isa"].draw("mpl", idle_wires=False, fold=-1)
# TODO (B): compare against a deliberately BAD layout (e.g. initial_layout on the worst qubits) to show it matters.

*TODO (B): interpretation. Depth/2q gates for JW vs tapered, and why the chosen qubits won.*

# 6. Noisy simulation and error mitigation
- **Owner:** B
- **Scores on:** Validation, Hardware awareness
- **Timebox:** 0:50–1:25

We optimise parameters noiselessly (Section 3), then **evaluate** them under the backend's noise model. That separates noise error from optimisation error. ZNE: fold each 2-qubit gate (scale factors 1, 3, 5) and extrapolate to zero noise.

In [ ]:
est = noisy_estimator(backend)
SCALES = [1, 3, 5]
rows = []
for d in HW_DISTANCES:
    pp = prepare_point(d, mapping="tapered")
    isa = best_transpile(pp.circuit, backend, n_seeds=10)["isa"]
    ev, sd = estimate([fold_two_qubit_gates(isa, s) for s in SCALES], pp.qubit_op, est, precision=0.002)
    E = ev + pp.nuclear_repulsion
    rows.append({"distance_A": d, "exact": pp.e_exact, "vqe_noiseless": pp.e_vqe,
                 "noisy": E[0], "noisy_std": sd[0],
                 "zne_linear": zne_extrapolate(SCALES, E, 1), "zne_quad": zne_extrapolate(SCALES, E, 2),
                 **{f"scale_{s}": e for s, e in zip(SCALES, E)}})
noisy = pd.DataFrame(rows)
for c in ["noisy", "zne_linear", "zne_quad"]:
    noisy[f"{c}_err"] = (noisy[c] - noisy.exact).abs()
noisy.to_csv(RESULTS / "pes_noisy_sim.csv", index=False); noisy

In [ ]:
# TODO (B): plot noisy vs ZNE vs exact; also an energy-vs-scale plot at one distance showing the extrapolation.
# Extra: repeat for the JW encoding at r_eq to show how much worse the deeper circuit is.

*TODO (B): interpretation. Is ZNE within chemical accuracy? Is linear or quadratic better, and does the extrapolation amplify shot noise (error bars)?*

# 7. Real quantum hardware
- **Owner:** B
- **Scores on:** Validation, Hardware awareness
- **Timebox:** submit by ~0:45, analyse 1:25–1:45

**Submit early:** queue times are unpredictable. Send **all distances × all ZNE scales as one job** (a single list of PUBs), which needs only one queue wait.

Error suppression and mitigation used: dynamical decoupling (XpXm), Pauli twirling on gates and measurements, TREX readout mitigation (`resilience_level=1`), plus our own folding-based ZNE on top.

Set your IBM Quantum account up once in a terminal (keep the token out of the notebook):
```python
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="...", instance="...")
```

In [ ]:
HW_FILE = RESULTS / "hardware_pes.csv"
if RUN_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService()
    hw_backend = service.least_busy(operational=True, simulator=False, min_num_qubits=5)
    print("Running on", hw_backend.name)
    points = [prepare_point(d, mapping="tapered") for d in HW_DISTANCES]
    pubs, meta = [], []
    for pp in points:
        isa = best_transpile(pp.circuit, hw_backend, n_seeds=20)["isa"]
        for s in SCALES:
            c = fold_two_qubit_gates(isa, s)
            pubs.append((c, pp.qubit_op.apply_layout(c.layout))); meta.append((pp, s))
    job = runtime_estimator(hw_backend).run(pubs)
    print("job id:", job.job_id())  # paste into the appendix
    res = job.result()
    rows = []
    for d in HW_DISTANCES:
        idx = [i for i, (pp, s) in enumerate(meta) if pp.distance == d]
        E = np.array([float(res[i].data.evs) for i in idx]) + meta[idx[0]][0].nuclear_repulsion
        pp = meta[idx[0]][0]
        rows.append({"distance_A": d, "exact": pp.e_exact, "hw_raw": E[0],
                     "hw_zne_linear": zne_extrapolate(SCALES, E, 1), "backend": hw_backend.name,
                     "job_id": job.job_id(), **{f"scale_{s}": e for s, e in zip(SCALES, E)}})
    pd.DataFrame(rows).to_csv(HW_FILE, index=False)

hardware = pd.read_csv(HW_FILE) if HW_FILE.exists() else None
hardware

In [ ]:
# TODO (B): hardware vs noisy-sim vs exact plot; errors vs chemical accuracy.

*TODO (B): interpretation. Does the noisy simulator predict hardware well? What did each mitigation layer buy?*

# 8. Performance and limitations
- **Owner:** A (writes) + B (numbers)
- **Scores on:** Performance and limitations
- **Timebox:** 1:25–1:45

### Error budget at equilibrium
| Source | Size (Ha) | Evidence |
|---|---|---|
| Basis set (STO-3G vs 6-31G) | *TODO* | Section 4 |
| Ansatz / optimiser (VQE vs exact, noiseless) | *TODO* | Section 3 |
| Shot noise (std at chosen precision) | *TODO* | Section 6 |
| Gate and readout noise (raw noisy − noiseless) | *TODO* | Sections 6 / 7 |
| Residual after mitigation (ZNE − exact) | *TODO* | Sections 6 / 7 |

### Combined figure
*TODO: one plot with exact, noiseless VQE, noisy sim (raw and ZNE) and hardware (raw and ZNE) vs distance. This is the headline figure.*

In [ ]:
# TODO: combined headline figure from results/*.csv

### Limitations and scaling
*TODO (both): be honest and specific. For example:*
- *Fixed-parameter evaluation: we never ran the optimisation loop on hardware. What would change with SPSA under noise?*
- *Tapering makes HeH⁺/STO-3G trivially small; how do qubits and 2q gates grow with basis (use Section 4 numbers)?*
- *ZNE: assumptions (noise scales linearly with folding), variance amplification, and cost (3× circuits).*
- *A noise-aware layout uses calibration data that drifts.*

# 9. Conclusions
- **Owner:** both
- **Scores on:** Presentation
- **Timebox:** 1:45–2:00

*TODO: 4–6 bullets with numbers. Then 2–3 bullets on what we'd do with more time.*

# Appendix
- **Hardware:** backend *TODO*, job ID(s) *TODO*, date *TODO*
- **Environment:** see the cell below.
- **Credits:** builds on B. Tirado, *Ground-State Energy of a Molecule with VQE*, BasQ Qiskit Fall Fest 2026 tutorial (CC BY 4.0 / Apache 2.0).

In [ ]:
from importlib.metadata import version
import sys
print("python", sys.version.split()[0])
for p in ["qiskit", "qiskit-nature", "qiskit-algorithms", "qiskit-aer", "qiskit-ibm-runtime", "pyscf"]:
    print(f"{p:20s}{version(p)}")